# Gram-Type Classification

In this we apply both the Radial and Strip conformal envelope methods to the real phage data to separate gram-negative from gram-positive host phages.

### Method
1. Load `host_type_all_preds.csv` (protein-level gram-type scores, 40 function columns) and `metadata.csv`
2. Join on `proteinID` to attach gram-type labels
3. Max-pool protein scores to phage level (one 40-dim score vector per phage)
4. NaN handling: drop columns with >50% NaN; impute remaining NaNs with column median from training data only (never zero — NaN means the model was not applicable, not zero evidence)
5. Split using the split column: 0 = test, 1–4 = train
6. Build one conformal envelope per gram type from training infectors (both methods)
7. Evaluate coverage, efficiency, and accuracy on the test set
8. Compare Radial vs Strip

## Decision rule
- Score vector inside gram-neg envelope → phage predicted to infect gram-negative host
- Score vector inside gram-pos envelope → phage predicted to infect gram-positive host
- Both → uncertain (prediction set size 2)
- Neither → empty prediction set

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(13)

### Loading the Data

- `host_type_all_preds.csv`: one row per protein, 40 function-specific gram-type score columns
- `metadata.csv`: one row per protein — `accession` (phage ID), `host_type` (gram-neg / gram-pos), `split` (0 = test, 1–4 = train)

Both files share `proteinID` as index.

In [2]:
SCORES_PATH = "../Data/host_type_all_preds.csv"
METADATA_PATH = "../Data/metadata.csv"

print("Loading scores...")
t0 = time.time()
scores_df = pd.read_csv(SCORES_PATH, index_col="proteinID")
print(f"  scores shape : {scores_df.shape}  ({time.time()-t0:.1f}s)")

print("Loading metadata...")
t0 = time.time()
meta_df = pd.read_csv(METADATA_PATH, index_col="proteinID")
print(f"  metadata shape : {meta_df.shape}  ({time.time()-t0:.1f}s)")

meta_df = meta_df[["accession", "host_type", "split"]]

Loading scores...
  scores shape : (1853074, 40)  (2.6s)
Loading metadata...
  metadata shape : (1853074, 8)  (2.5s)


### Joining the Scores and Metadata

In [3]:
combined   = scores_df.join(meta_df, how="inner")
score_cols = [c for c in scores_df.columns if c.startswith("host_type_")]

print(f"Combined shape: {combined.shape}")
print(f"Score columns : {len(score_cols)}")
print("\nGram-type distribution (protein level):")
print(combined["host_type"].value_counts())
print("\nSplit distribution (0=test, 1–4=train):")
print(combined["split"].value_counts().sort_index())

Combined shape: (1853074, 43)
Score columns : 40

Gram-type distribution (protein level):
host_type
gram-neg    1151735
gram-pos     699623
unknown        1716
Name: count, dtype: int64

Split distribution (0=test, 1–4=train):
split
0    408069
1    313252
2    373117
3    371171
4    387465
Name: count, dtype: int64


### Max-Pool Proteins → Phage Level Scores

For each phage and each function column, take the maximum score across all its proteins.

**Why max-pooling?**  
If any single protein strongly signals a gram-type, that is evidence for the whole phage.

**NaN semantics:**  
A protein gets a score for a function only if it was annotated with that function.  
NaN means "this model was not applicable" — not zero evidence.  
NaN remains NaN after max-pooling only if ALL proteins in the phage have NaN for that function.  

In [4]:
print("Max-pooling proteins → phage level...")
t0 = time.time()

phage_scores = combined.groupby("accession")[score_cols].max()   # NaN-aware
phage_meta   = combined.groupby("accession")[["host_type", "split"]].first()
phage_df     = phage_scores.join(phage_meta)

print(f"  Phage-level shape: {phage_df.shape}  ({time.time()-t0:.1f}s)")
print("\nGram-type distribution (phage level):")
print(phage_df["host_type"].value_counts())
print("\nSplit distribution (phage level):")
print(phage_df["split"].value_counts().sort_index())

Max-pooling proteins → phage level...
  Phage-level shape: (18474, 42)  (0.6s)

Gram-type distribution (phage level):
host_type
gram-neg    10735
gram-pos     7708
unknown        31
Name: count, dtype: int64

Split distribution (phage level):
split
0    4433
1    3173
2    3646
3    3588
4    3634
Name: count, dtype: int64


### NaN Analysis and Column Filtering

Before imputation, we inspect how many phages have NaN for each function column.  
Columns where >50% of phages have NaN carry almost no signal and are dropped.

In [5]:
nan_rates = phage_df[score_cols].isna().mean().sort_values(ascending=False)

print(f"NaN rate per column (top 10):")
print(nan_rates.head(10).to_string())

NAN_THRESHOLD = 0.50
valid_cols = nan_rates[nan_rates <= NAN_THRESHOLD].index.tolist()
dropped    = nan_rates[nan_rates  > NAN_THRESHOLD].index.tolist()

print(f"\nColumns kept   : {len(valid_cols)} / {len(score_cols)}")
print(f"Columns dropped: {len(dropped)}")
if dropped:
    print(f"  Dropped: {dropped}")

K = len(valid_cols)
print(f"\nFinal K (dimensions) = {K}")

NaN rate per column (top 10):
host_type_sir2                         0.958320
host_type_lysis_inhibitor              0.893255
host_type_toxin                        0.795821
host_type_ejection                     0.767078
host_type_super_infection              0.744181
host_type_transcriptional_activator    0.740500
host_type_tail_sheath                  0.721176
host_type_replication_initiation       0.720418
host_type_reductase                    0.637869
host_type_annealing                    0.550936

Columns kept   : 27 / 40
Columns dropped: 13
  Dropped: ['host_type_sir2', 'host_type_lysis_inhibitor', 'host_type_toxin', 'host_type_ejection', 'host_type_super_infection', 'host_type_transcriptional_activator', 'host_type_tail_sheath', 'host_type_replication_initiation', 'host_type_reductase', 'host_type_annealing', 'host_type_spanin', 'host_type_baseplate', 'host_type_val']

Final K (dimensions) = 27


### Train / Test Split and Median Imputation

NaN means the model was not applicable to that protein/phage — not that the score is zero.  
Filling with zero would place these phages at the origin of score space, far from where true infectors live, which would distort the envelope and break coverage guarantees.

**Imputation protocol:**  
1. Compute **column medians from the training set only** (no data leakage)
2. Apply those medians to fill NaNs in both train and test sets

In [6]:
# --- 1. Train / Test Split ---
# split=0  → test  |  split=1,2,3,4 → train
train_df = phage_df[phage_df["split"] != 0].copy()
test_df  = phage_df[phage_df["split"] == 0].copy()

# Filter gram types (keeping consistency with your original logic)
train_df = train_df[train_df["host_type"].isin(["gram-neg", "gram-pos"])]
test_df  = test_df[test_df["host_type"].isin(["gram-neg", "gram-pos"])]

print(f"Train phages : {len(train_df)}")
print(f"Test  phages : {len(test_df)}")

# --- 2. Feature Engineering (Missingness Signal) ---
# We calculate this BEFORE imputation, otherwise the NaNs disappear!
train_df['nan_count'] = train_df[valid_cols].isna().sum(axis=1)
test_df['nan_count']  = test_df[valid_cols].isna().sum(axis=1)

# --- 3. Median Imputation with Safety Cap ---
train_medians = train_df[valid_cols].median()

# If a median is > 0.6, we cap it at 0.5 to keep it neutral
safe_impute_values = train_medians.copy()
safe_impute_values[safe_impute_values > 0.6] = 0.5 

# Apply the safe imputation
train_df[valid_cols] = train_df[valid_cols].fillna(safe_impute_values)
test_df [valid_cols] = test_df [valid_cols].fillna(safe_impute_values)

# Residual NaNs (columns entirely NaN in training) → fill with 0.5
train_df[valid_cols] = train_df[valid_cols].fillna(0.5)
test_df [valid_cols] = test_df [valid_cols].fillna(0.5)

# --- 4. Reporting & Extraction ---
print("\nMedians summary after safety cap:")
print(safe_impute_values.describe())

print(f"\nResidual NaNs after imputation — train: {train_df[valid_cols].isna().sum().sum()}")
print(f"Residual NaNs after imputation — test : {test_df[valid_cols].isna().sum().sum()}")

# Extract score matrices (Note: you might want to include 'nan_count' in your final features)
train_gramneg = train_df[train_df["host_type"] == "gram-neg"][valid_cols].values
train_grampos = train_df[train_df["host_type"] == "gram-pos"][valid_cols].values

test_scores = test_df[valid_cols].values
test_labels = test_df["host_type"].values

print(f"\nTraining infectors:")
print(f"  gram-neg : {train_gramneg.shape}")
print(f"  gram-pos : {train_grampos.shape}")

Train phages : 14021
Test  phages : 4422

Medians summary after safety cap:
count    27.000000
mean      0.334596
std       0.193141
min       0.008645
25%       0.185373
50%       0.408910
75%       0.500000
max       0.570021
dtype: float64

Residual NaNs after imputation — train: 0
Residual NaNs after imputation — test : 0

Training infectors:
  gram-neg : (7924, 27)
  gram-pos : (6097, 27)


In [7]:
# # --- Train / test split ---
# # split=0  → test  |  split=1,2,3,4 → train
# train_df = phage_df[phage_df["split"] != 0].copy()
# test_df  = phage_df[phage_df["split"] == 0].copy()

# # Drop 'unknown' gram type (only 31 phages at phage level)
# train_df = train_df[train_df["host_type"].isin(["gram-neg", "gram-pos"])]
# test_df  = test_df [test_df ["host_type"].isin(["gram-neg", "gram-pos"])]

# print(f"Train phages : {len(train_df)}")
# print(f"Test  phages : {len(test_df)}")

# # --- Median imputation (training medians only) ---
# train_medians = train_df[valid_cols].median()

# train_df[valid_cols] = train_df[valid_cols].fillna(train_medians)
# test_df [valid_cols] = test_df [valid_cols].fillna(train_medians)

# # Residual NaNs (column entirely NaN in training) → fill with 0.5 as a safe neutral
# train_df[valid_cols] = train_df[valid_cols].fillna(0.5)
# test_df [valid_cols] = test_df [valid_cols].fillna(0.5)

# print(f"Residual NaNs after imputation — train: {train_df[valid_cols].isna().sum().sum()}")
# print(f"Residual NaNs after imputation — test : {test_df [valid_cols].isna().sum().sum()}")

# # --- Extract score matrices per gram type ---
# train_gramneg = train_df[train_df["host_type"] == "gram-neg"][valid_cols].values
# train_grampos = train_df[train_df["host_type"] == "gram-pos"][valid_cols].values

# test_scores = test_df[valid_cols].values
# test_labels = test_df["host_type"].values

# print(f"\nTraining infectors:")
# print(f"  gram-neg : {train_gramneg.shape}")
# print(f"  gram-pos : {train_grampos.shape}")

### Envelope Methods

Both methods follow the same two-stage calibration:
- **S1 (shape discovery)**: learns the shape of the envelope
- **S2 (size scaling)**: finds the global scale factor t̂ that gives exactly 1−α coverage

#### Method 1 — Radial (Angular / Sector)
Sample M random directions on the positive orthant of the unit sphere.  
For each direction, compute a local radius threshold from nearby calibration points.  
A test point is inside if it fits inside at least one sector.

#### Method 2 — Strip (Vertical / Horizontal)
Slice score space into M equal-width bins along each dimension.  
Within each bin, compute quantile thresholds for all other dimensions.  
A test point is inside if all pairwise dimension constraints hold.

In [8]:
def split_data(scores, split_ratio=0.5):
    """Randomly split scores into S1 (shape discovery) and S2 (size scaling)."""
    n = len(scores)
    idx = np.random.permutation(n)
    cut = int(n * split_ratio)
    return scores[idx[:cut]], scores[idx[cut:]]

#### Radial Method

In [9]:
def sample_positive_sphere(M, K):
    """
    Sample M directions uniformly on the positive orthant of the unit sphere S^{K-1}.
    Draw from N(0,I), take absolute values, normalise.
    """
    V = np.abs(np.random.randn(M, K))
    norms = np.linalg.norm(V, axis=1, keepdims=True)
    return V / (norms + 1e-12)

In [10]:
def radial_shape_discovery(S1, alpha, M, delta_deg=15):
    """
    For each of M directions on the positive unit sphere:
      1. Find S1 points whose direction is within delta_deg of it
      2. Compute the (1-alpha)
    """
    K = S1.shape[1]
    U = sample_positive_sphere(M, K)
    mags = np.linalg.norm(S1, axis=1)
    dirs = S1 / (mags[:, None] + 1e-12)
    cos_thresh = np.cos(np.radians(delta_deg))
    q_tilde = np.zeros(M)

    for m in range(M):
        sims = dirs @ U[m]
        mask = sims >= cos_thresh
        if mask.sum() < 5:
            q_tilde[m] = np.quantile(mags, 1 - alpha)   # fallback: global quantile
        else:
            q_tilde[m] = np.quantile(mags[mask], 1 - alpha)

    return U, q_tilde

In [11]:
def radial_size_scaling(S2, U, q_tilde, alpha):
    """
    Find t_hat: the global scale factor giving the envelope exactly (1-alpha)
    coverage on S2.

    tau(s) = min_m { max_k s[k] / boundary[m,k] }
    t_hat  = (1-alpha) quantile of tau scores over S2.
    """
    boundary = U * q_tilde[:, None]                            # (M, K)
    ratios = S2[:, None, :] / (boundary[None, :, :] + 1e-12)# (N2, M, K)
    t_per_sector = ratios.max(axis=2)                              # (N2, M)
    tau_scores = t_per_sector.min(axis=1)                        # (N2,)

    n2 = len(tau_scores)
    idx = int(np.ceil((n2 + 1) * (1 - alpha))) - 1
    t_hat = np.sort(tau_scores)[np.clip(idx, 0, n2 - 1)]
    return t_hat

In [12]:
def radial_is_in_region(scores, envelope):
    """
    True for each score vector inside the scaled radial envelope.
    A point is inside if ANY sector contains it (all K dims ≤ scaled boundary).
    """
    U, q_tilde, t_hat = envelope["U"], envelope["q_tilde"], envelope["t_hat"]
    q_final = q_tilde * t_hat
    boundary = U * q_final[:, None]                                # (M, K)
    inside = np.any(
        np.all(scores[:, None, :] <= boundary[None, :, :], axis=2), axis=1
    )
    return inside

In [13]:
def radial_build_envelope(infection_scores, alpha, M=200, delta_deg=15):
    """Full two-stage calibration for the radial method."""
    S1, S2  = split_data(infection_scores)
    U, q_tilde = radial_shape_discovery(S1, alpha, M, delta_deg)
    t_hat = radial_size_scaling(S2, U, q_tilde, alpha)
    return {"U": U, "q_tilde": q_tilde, "t_hat": t_hat, "method": "radial"}

#### Strip Method

In [14]:
def strip_shape_discovery(S1, alpha, M):
    """
    For each conditioning dimension j and each of M equal-width bins along j:
      - Find S1 points in that bin
      - Compute (1-alpha) quantile of every other dimension i
      → limits[j, i, m] = threshold for dim i when dim j is in bin m

    Monotonicity propagation ensures the envelope has no holes.

    Returns
    -------
    bin_edges : (K, M+1) bin boundaries
    limits : (K, K, M) quantile thresholds
    """
    N1, K = S1.shape

    bin_edges = np.zeros((K, M + 1))
    for j in range(K):
        bin_edges[j] = np.linspace(S1[:, j].min(), S1[:, j].max(), M + 1)

    limits = np.zeros((K, K, M))
    for j in range(K):
        for m in range(M):
            lo, hi   = bin_edges[j, m], bin_edges[j, m + 1]
            in_strip = (S1[:, j] >= lo) & (S1[:, j] < hi)
            for i in range(K):
                if i == j:
                    continue
                if in_strip.sum() < 3:
                    limits[j, i, m] = np.quantile(S1[:, i], 1 - alpha)
                else:
                    limits[j, i, m] = np.quantile(S1[in_strip, i], 1 - alpha)

    # Monotonicity: limits are non-decreasing as bin index decreases
    for j in range(K):
        for i in range(K):
            if i == j:
                continue
            for m in range(M - 2, -1, -1):
                limits[j, i, m] = max(limits[j, i, m], limits[j, i, m + 1])

    return bin_edges, limits

In [15]:
def get_bin_indices(scores, bin_edges):
    """For each point and dimension, find which bin it falls in."""
    N, K = scores.shape
    M = bin_edges.shape[1] - 1
    indices = np.zeros((N, K), dtype=int)
    for j in range(K):
        raw = np.digitize(scores[:, j], bin_edges[j]) - 1
        indices[:, j] = np.clip(raw, 0, M - 1)
    return indices

In [16]:
def strip_size_scaling(S2, bin_edges, limits, alpha):
    """
    Find t_hat for the strip envelope.

    tau(s) = max over (j, i≠j) of: s[i] / limits[j, i, bin_j(s)]
    t_hat = (1-alpha) quantile of tau scores over S2.
    """
    N2, K = S2.shape
    bin_idx = get_bin_indices(S2, bin_edges)

    ratios = np.zeros((N2, K, K))
    for j in range(K):
        for i in range(K):
            if i == j:
                continue
            lims = limits[j, i, bin_idx[:, j]]
            ratios[:, j, i] = S2[:, i] / (lims + 1e-12)

    tau_scores = ratios.max(axis=(1, 2))
    n2 = N2
    idx = int(np.ceil((n2 + 1) * (1 - alpha))) - 1
    t_hat = np.sort(tau_scores)[np.clip(idx, 0, n2 - 1)]
    return t_hat

In [17]:
def strip_is_in_region(scores, envelope):
    """
    True for each score vector inside the scaled strip envelope.
    A point is inside if for ALL pairs (j, i≠j): s[i] ≤ t_hat * limits[j,i,bin_j(s)]
    """
    bin_edges, limits, t_hat = (
        envelope["bin_edges"], envelope["limits"], envelope["t_hat"]
    )
    N, K = scores.shape
    bin_idx = get_bin_indices(scores, bin_edges)
    scaled_limits = limits * t_hat

    inside = np.ones(N, dtype=bool)
    for j in range(K):
        for i in range(K):
            if i == j:
                continue
            lims = scaled_limits[j, i, bin_idx[:, j]]
            inside &= (scores[:, i] <= lims)
    return inside

In [18]:
def strip_build_envelope(infection_scores, alpha, M=20):
    """Full two-stage calibration for the strip method."""
    S1, S2 = split_data(infection_scores)
    bin_edges, limits = strip_shape_discovery(S1, alpha, M)
    t_hat  = strip_size_scaling(S2, bin_edges, limits, alpha)
    return {"bin_edges": bin_edges, "limits": limits, "t_hat": t_hat, "method": "strip"}

In [19]:
def is_in_region(scores, envelope):
    if envelope["method"] == "radial":
        return radial_is_in_region(scores, envelope)
    else:
        return strip_is_in_region(scores, envelope)


def predict_gram_type(test_scores, envelopes):
    """
    For each test phage, check its score vector against both gram-type envelopes
    and return the prediction set.
    """
    in_gramneg = is_in_region(test_scores, envelopes["gram-neg"])
    in_grampos = is_in_region(test_scores, envelopes["gram-pos"])

    results = []
    for i in range(len(test_scores)):
        pred_set = []
        if in_gramneg[i]: pred_set.append("gram-neg")
        if in_grampos[i]: pred_set.append("gram-pos")
        results.append({
            "in_gramneg"    : bool(in_gramneg[i]),
            "in_grampos"    : bool(in_grampos[i]),
            "prediction_set": pred_set,
        })
    return results

### Building the Envelopes on Training Data

In [20]:
alpha = 0.1   # target coverage = 1 - alpha = 0.90

print("Building radial envelopes...")
t0 = time.time()
radial_envelopes = {"gram-neg": radial_build_envelope(train_gramneg, alpha, M=200, delta_deg=30),
                    "gram-pos": radial_build_envelope(train_grampos, alpha, M=200, delta_deg=30),}
print(f"  Done in {time.time()-t0:.1f}s")
print(f"  t_hat (gram-neg) = {radial_envelopes['gram-neg']['t_hat']:.4f}")
print(f"  t_hat (gram-pos) = {radial_envelopes['gram-pos']['t_hat']:.4f}")

print("\nBuilding strip envelopes...")
t0 = time.time()
strip_envelopes = {"gram-neg": strip_build_envelope(train_gramneg, alpha, M=20),
                   "gram-pos": strip_build_envelope(train_grampos, alpha, M=20),}
print(f"  Done in {time.time()-t0:.1f}s")
print(f"  t_hat (gram-neg) = {strip_envelopes['gram-neg']['t_hat']:.4f}")
print(f"  t_hat (gram-pos) = {strip_envelopes['gram-pos']['t_hat']:.4f}")

Building radial envelopes...
  Done in 0.2s
  t_hat (gram-neg) = 3.4695
  t_hat (gram-pos) = 4.3962

Building strip envelopes...
  Done in 2.1s
  t_hat (gram-neg) = 2.0952
  t_hat (gram-pos) = 1.0034


### Evaluating on the test set

For each test phage we check:
- **Coverage**: is the true gram type in the prediction set? (should be ≥ 1−α = 0.90)
- **Efficiency**: how large is the prediction set on average? (smaller = more informative)
- **Empty rate**: fraction of phages with no prediction (bad — means the envelope is too tight)
- **Singleton accuracy**: when set size = 1, is it correct?
- **Per-gram-type coverage**: coverage broken down by true gram type

In [21]:
def evaluate(results, true_labels, method_name, alpha):
    n          = len(results)
    set_sizes  = [len(r["prediction_set"]) for r in results]

    # Overall coverage
    covered = sum(true_labels[i] in r["prediction_set"] for i, r in enumerate(results))

    # Per-gram-type coverage
    for gt in ["gram-neg", "gram-pos"]:
        idx_gt  = [i for i, lbl in enumerate(true_labels) if lbl == gt]
        cov_gt  = sum(true_labels[i] in results[i]["prediction_set"] for i in idx_gt)
        n_gt    = len(idx_gt)

    # Singleton accuracy
    singletons = [(r, true_labels[i]) for i, r in enumerate(results) if len(r["prediction_set"]) == 1]
    correct_singletons = sum(r["prediction_set"][0] == lbl for r, lbl in singletons)

    empty = sum(s == 0 for s in set_sizes)

    print(f"\n{'='*58}")
    print(f" {method_name.upper()} METHOD — {n} test phages | alpha={alpha}")
    print(f"{'='*58}")
    print(f" Coverage (true type in set) : {covered/n:.3f}  (target ≥ {1-alpha:.2f})")
    print(f" Average prediction set size : {np.mean(set_sizes):.3f}")
    print(f" Empty prediction sets: {empty} ({empty/n*100:.1f}%)")
    if singletons:
        print(f" Singleton accuracy : {correct_singletons/len(singletons):.3f}  ({len(singletons)} singletons)")
    else:
        print(" Singleton accuracy : N/A (no singletons)")

    print(f"\n Per-gram-type coverage:")
    for gt in ["gram-neg", "gram-pos"]:
        idx_gt = [i for i, lbl in enumerate(true_labels) if lbl == gt]
        cov_gt = sum(true_labels[i] in results[i]["prediction_set"] for i in idx_gt)
        n_gt   = len(idx_gt)
        print(f"   {gt:<12}: {cov_gt/n_gt:.3f}  (n={n_gt})")

    return {
        "coverage" : covered / n,
        "avg_set_size" : np.mean(set_sizes),
        "empty_rate" : empty / n,
        "singleton_rate" : len(singletons) / n,
        "singleton_accuracy": correct_singletons / len(singletons) if singletons else None,
        "set_sizes": set_sizes,
        "per_type_coverage" : {
            gt: sum(true_labels[i] in results[i]["prediction_set"]
                    for i in [j for j, lbl in enumerate(true_labels) if lbl == gt])
               / max(1, sum(lbl == gt for lbl in true_labels))
            for gt in ["gram-neg", "gram-pos"]
        },
    }

In [22]:
radial_results = predict_gram_type(test_scores, radial_envelopes)
strip_results  = predict_gram_type(test_scores, strip_envelopes)

radial_metrics = evaluate(radial_results, test_labels, "Radial", alpha)
strip_metrics  = evaluate(strip_results,  test_labels, "Strip",  alpha)


 RADIAL METHOD — 4422 test phages | alpha=0.1
 Coverage (true type in set) : 0.891  (target ≥ 0.90)
 Average prediction set size : 1.527
 Empty prediction sets: 320 (7.2%)
 Singleton accuracy : 0.887  (1453 singletons)

 Per-gram-type coverage:
   gram-neg    : 0.942  (n=2811)
   gram-pos    : 0.801  (n=1611)

 STRIP METHOD — 4422 test phages | alpha=0.1
 Coverage (true type in set) : 0.900  (target ≥ 0.90)
 Average prediction set size : 1.537
 Empty prediction sets: 180 (4.1%)
 Singleton accuracy : 0.846  (1688 singletons)

 Per-gram-type coverage:
   gram-neg    : 0.908  (n=2811)
   gram-pos    : 0.888  (n=1611)


### 9 — Summary Table

In [ ]:
summary = pd.DataFrame({
    "Method" : ["Radial", "Strip"],
    "Coverage": [radial_metrics["coverage"], strip_metrics["coverage"]],
    "Target coverage": [1 - alpha, 1 - alpha],
    "Avg set size": [radial_metrics["avg_set_size"], strip_metrics["avg_set_size"]],
    "Empty rate" : [radial_metrics["empty_rate"], strip_metrics["empty_rate"]],
    "Singleton accuracy": [radial_metrics["singleton_accuracy"], strip_metrics["singleton_accuracy"]],
})
print(summary.to_string(index=False))

Method  Coverage  Target coverage  Avg set size  Empty rate  Singleton accuracy
Radial  0.890547              0.9      1.526685    0.072365            0.887130
 Strip  0.900498              0.9      1.536861    0.040706            0.845972


In [29]:
methods = ["Radial", "Strip"]
metrics = [radial_metrics, strip_metrics]

set_size_data = {}

for name, m in zip(methods, metrics):
    sizes = m["set_sizes"]
    n = len(sizes)
    
    # Calculating the fractions for Empty (0), Size 1, and Size 2 predictions
    distribution = {
        "Empty":  sizes.count(0) / n,
        "Size 1": sizes.count(1) / n,
        "Size 2": sizes.count(2) / n
    }
    set_size_data[name] = distribution


print(f"--- Set Size Distribution (alpha={alpha}) ---")
for method, counts in set_size_data.items():
    print(f"\n{method} Method:")
    for label, fraction in counts.items():
        print(f" {label:<8}: {fraction*100:.2f} %")

--- Set Size Distribution (alpha=0.1) ---

Radial Method:
 Empty   : 7.24 %
 Size 1  : 32.86 %
 Size 2  : 59.91 %

Strip Method:
 Empty   : 4.07 %
 Size 1  : 38.17 %
 Size 2  : 57.76 %
